<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session03-lab.ipynb" target="_blank" rel="noopener noreferrer" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open published copy in Google Colab</a>

# Session 3: Biological Databases

**Lab notebook**

Build a dossier on human myoglobin (*MB*). Use the sequence cells and partner websites for Questions 1–2. Return to the structure lookup for Question 6. The Canvas instructions guide you through the genome browser, protein annotations, the structure paper and interaction evidence.

This notebook needs internet access and a Python 3 kernel. You can run the downloaded notebook in Jupyter or VS Code. In Colab you can upload the downloaded file directly. [Course setup](https://course.bioinfo.se/setup.html) · [Session 3 reading](https://course.bioinfo.se/session03.html)

## Setup

The first cell installs Biopython and Requests only when they are missing. Biopython reads the annotated sequence files and translates the coding region.

In [1]:
import importlib.util
import subprocess
import sys

packages = {"biopython": "Bio", "requests": "requests"}
missing = [package for package, module in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [2]:
from datetime import datetime, timezone
from io import StringIO
import time

import requests
from Bio import SeqIO
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry temporary server errors and respect the server's retry delay.
http = requests.Session()
retries = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=(429, 502, 503, 504),
    allowed_methods={"GET"},
)
http.mount("https://", HTTPAdapter(max_retries=retries))

efetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
retrieved = datetime.now(timezone.utc)
print("Retrieval started (UTC):", retrieved.isoformat(timespec="seconds"))

Retrieval started (UTC): 2026-11-13T16:39:55+00:00


## Question 1 — One archived record, three interfaces

Retrieve `X00371.1`, an archived human myoglobin **genomic DNA** record, through NCBI, ENA and DDBJ. The partners provide different file formats, so we parse each record before comparing the nucleotide strings.

Before running the code, predict which fields should agree. Then open the DDBJ website and one of the other two websites. Find the accession and version, organism and length on both pages. [DDBJ](https://getentry.ddbj.nig.ac.jp/getentry/na/X00371.1/?filetype=html) · [NCBI Nucleotide](https://www.ncbi.nlm.nih.gov/nuccore/X00371.1) · [ENA](https://www.ebi.ac.uk/ena/browser/view/X00371.1)

In [3]:
archive_accession = "X00371.1"
archive_sources = {
    "NCBI": (
        efetch,
        {"db": "nuccore", "id": archive_accession, "rettype": "gb", "retmode": "text"},
        "genbank",
    ),
    "ENA": (
        f"https://www.ebi.ac.uk/ena/browser/api/embl/{archive_accession}",
        {"download": "false"},
        "embl",
    ),
    "DDBJ": (
        f"https://getentry.ddbj.nig.ac.jp/getentry/na/{archive_accession}/",
        {"filetype": "text", "format": "flatfile"},
        "genbank",
    ),
}

archive_records = {}
for partner, (url, params, file_format) in archive_sources.items():
    response = http.get(url, params=params, timeout=30)
    response.raise_for_status()
    archive_records[partner] = SeqIO.read(StringIO(response.text), file_format)

In [4]:
def taxonomy_id(record):
    """Read the taxonomy ID from the record's source feature."""
    return next(
        xref.split(":", 1)[1]
        for feature in record.features if feature.type == "source"
        for xref in feature.qualifiers.get("db_xref", [])
        if xref.startswith("taxon:")
    )

print(f"{'Partner':8s} {'Accession':12s} {'Organism':24s} {'Taxon':6s} {'Bases':>6s}")
for partner, record in archive_records.items():
    print(f"{partner:8s} {record.id:12s} {record.annotations['organism']:24s} "
          f"{taxonomy_id(record):6s} {len(record.seq):6d}")

reference = str(archive_records["NCBI"].seq).upper()
print("All accession versions match the request:",
      all(record.id == archive_accession for record in archive_records.values()))
print("All taxonomy IDs identify humans:",
      all(taxonomy_id(record) == "9606" for record in archive_records.values()))
print("All sequence lengths agree:",
      all(len(record.seq) == len(reference) for record in archive_records.values()))
print("All nucleotide strings agree:",
      all(str(record.seq).upper() == reference for record in archive_records.values()))

Partner  Accession    Organism                 Taxon   Bases
NCBI     X00371.1     Homo sapiens             9606     3768
ENA      X00371.1     Homo sapiens (human)     9606     3768
DDBJ     X00371.1     Homo sapiens             9606     3768
All accession versions match the request: True
All taxonomy IDs identify humans: True
All sequence lengths agree: True
All nucleotide strings agree: True


Keep the two record-page URLs in your notes. Use the sequence length, accession version and sequence comparison to select all supported statements in Question 1. Use the taxonomy ID to check organism identity if the names are formatted differently.

## Question 2 — From reference transcript to protein

Retrieve the myoglobin reference transcript `NM_005368.3`. Find its annotated coding sequence (CDS) and the RefSeq protein identifier in the CDS feature. The transcript also includes untranslated sequence, so the code translates the annotated CDS.

Biopython reports feature boundaries using zero-based, half-open coordinates. The displayed interval below is converted to one-based, inclusive positions on the transcript. [NCBI reference transcript](https://www.ncbi.nlm.nih.gov/nuccore/NM_005368.3)

In [5]:
transcript_accession = "NM_005368.3"
time.sleep(0.5)  # space requests to NCBI if the cells run in quick succession
response = http.get(
    efetch,
    params={"db": "nuccore", "id": transcript_accession,
            "rettype": "gb", "retmode": "text"},
    timeout=30,
)
response.raise_for_status()
transcript = SeqIO.read(StringIO(response.text), "genbank")
if transcript.id != transcript_accession or taxonomy_id(transcript) != "9606":
    raise ValueError("Check the returned transcript accession and organism.")

cds = next(feature for feature in transcript.features if feature.type == "CDS")
protein_accession = cds.qualifiers["protein_id"][0]
coding_sequence = cds.extract(transcript.seq)
translation_table = int(cds.qualifiers.get("transl_table", ["1"])[0])
translated = str(coding_sequence.translate(table=translation_table, cds=True))

print("Transcript:", transcript.id)
print("Molecule type:", transcript.annotations["molecule_type"])
print("Transcript length:", len(transcript.seq), "bases")
print("CDS interval on transcript (one-based, inclusive):",
      int(cds.location.start) + 1, "to", int(cds.location.end))
print("CDS length including stop codon:", len(coding_sequence), "bases")
print("Linked RefSeq protein:", protein_accession)
print("Translation length:", len(translated), "amino acids")
print("Translation agrees with RefSeq CDS annotation:",
      translated == cds.qualifiers["translation"][0])

Transcript: NM_005368.3
Molecule type: mRNA
Transcript length: 1070 bases
CDS interval on transcript (one-based, inclusive): 73 to 537
CDS length including stop codon: 465 bases
Linked RefSeq protein: NP_005359.1
Translation length: 154 amino acids
Translation agrees with RefSeq CDS annotation: True


### Check the UniProt cross-reference and sequence

Use the reviewed human myoglobin entry to check whether the precise RefSeq transcript–protein pair is cross-referenced. Then compare its amino-acid sequence with your CDS translation. `cds=True` checks a complete coding sequence and excludes the terminal stop codon from the translated string. [UniProtKB entry](https://www.uniprot.org/uniprotkb/P02144/entry)

In [6]:
uniprot_accession = "P02144"
response = http.get(
    f"https://rest.uniprot.org/uniprotkb/{uniprot_accession}.json",
    timeout=30,
)
response.raise_for_status()
entry = response.json()
if (entry["primaryAccession"] != uniprot_accession
        or entry["organism"]["taxonId"] != 9606):
    raise ValueError("Check the returned UniProt accession and organism.")

refseq_links = [link for link in entry.get("uniProtKBCrossReferences", [])
                if link["database"] == "RefSeq"]
matching_pair = any(
    link["id"] == protein_accession
    and any(prop["key"] == "NucleotideSequenceId" and prop["value"] == transcript_accession
            for prop in link.get("properties", []))
    for link in refseq_links
)
print("UniProt accession:", entry["primaryAccession"])
print("UniProt entry type:", entry["entryType"])
print("The specific transcript–protein pair is cross-referenced:", matching_pair)
print("CDS translation equals the UniProt sequence:",
      translated == entry["sequence"]["value"])

UniProt accession: P02144
UniProt entry type: UniProtKB reviewed (Swiss-Prot)
The specific transcript–protein pair is cross-referenced: True
CDS translation equals the UniProt sequence: True


Use the linked RefSeq protein accession and version, the translated length and the UniProt sequence comparison to choose your answer to Question 2. Continue with the Canvas website tasks using the same transcript and protein record. Sequence agreement checks the link between the records. The later tasks ask you to find evidence for individual protein claims.

## Question 6 — Follow a protein accession to two structure records

Start from the reviewed UniProtKB record retrieved above. List its experimental PDB cross-references, then check how PDB entry `3RGK` was determined. Use the same protein accession to find the predicted model in AlphaFold DB. Open both record pages to compare where the coordinates came from and what sequence or construct each record represents.

[UniProtKB](https://www.uniprot.org/uniprotkb/P02144/entry) · [RCSB PDB](https://www.rcsb.org/structure/3RGK) · [AlphaFold DB](https://alphafold.ebi.ac.uk/entry/P02144)

In [ ]:
pdb_links = sorted({link["id"] for link in entry.get("uniProtKBCrossReferences", [])
                    if link["database"] == "PDB"})
print(f"Experimental PDB entries linked from UniProtKB {uniprot_accession}:")
for pdb_link in pdb_links:
    print(f"  {pdb_link}: https://www.rcsb.org/structure/{pdb_link}")

pdb_id = "3RGK"
pdb_response = http.get(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}", timeout=30)
pdb_response.raise_for_status()
pdb_record = pdb_response.json()
methods = ", ".join(experiment["method"] for experiment in pdb_record["exptl"])
print(f"\n{pdb_id} experimental method: {methods}")

af_response = http.get(
    f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_accession}", timeout=30
)
af_response.raise_for_status()
prediction = next(
    (record for record in af_response.json()
     if record["uniprotAccession"] == uniprot_accession
     and record["uniprotStart"] == 1
     and record["uniprotEnd"] == len(entry["sequence"]["value"])),
    None,
)
if prediction is None:
    raise ValueError(f"No full-length AlphaFold DB model returned for {uniprot_accession}.")
print("AlphaFold DB model:", prediction["entryId"])
print("Reference protein accession:", prediction["uniprotAccession"])
print("Reference sequence positions covered:",
      prediction["uniprotStart"], "to", prediction["uniprotEnd"])

Use the notebook and the two websites to answer Question 6. The shared accession connects the records to the same protein. Open the PDB entry's primary citation for Question 7 to check which version of the protein was used in the experiment.